In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image
import albumentations as A
import time
import re
import hashlib


class IndustrialCompostAugmenter:
    def __init__(self, input_dirs, output_base_dir):
        self.input_dirs = [d.rstrip("/") for d in input_dirs]
        self.output_base_dir = output_base_dir.rstrip("/")

        self.transform_pipeline = A.Compose([
            A.Rotate(limit=90, p=0.5),
            A.HorizontalFlip(p=0.3),
            A.VerticalFlip(p=0.3),
            A.Affine(
                translate_percent=0.15,
                scale=0.15,
                rotate=30,
                shear=10,
                p=0.4,
            ),
            A.OpticalDistortion(distort_limit=0.3, p=0.3),
            A.CoarseDropout(
                num_holes_range=(1, 5),
                hole_height_range=(1, 24),
                hole_width_range=(1, 24),
                fill=0,
                p=0.3,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.08,
                contrast_limit=0.08,
                p=0.4,
            ),
        ])

    def sanitize_filename(self, filename):
        name_without_ext = os.path.splitext(filename)[0]

        name_without_ext = re.sub(r'[点]', '_', name_without_ext)
        name_without_ext = re.sub(r'[^\w\-_\. ]', '_', name_without_ext)

        name_without_ext = re.sub(r'_+', '_', name_without_ext)

        if len(name_without_ext) > 100:
            name_without_ext = name_without_ext[:100]

        return name_without_ext

    def read_image(self, img_path):
        try:
            pil_img = Image.open(img_path)
            if pil_img.mode != "RGB":
                pil_img = pil_img.convert("RGB")
            return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        except Exception as e:
            print(f"Failed to read image {img_path}: {str(e)}")
            return None

    def save_image_pil(self, image, output_path):
        try:
            if len(image.shape) == 3 and image.shape[2] == 3:
                pil_image = Image.fromarray(image)
            elif len(image.shape) == 2:
                pil_image = Image.fromarray(image)
            else:
                pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            pil_image.save(output_path, quality=95)
            return True
        except Exception as e:
            print(f"PIL save failed {output_path}: {str(e)}")
            return False

    def save_image_cv2(self, image, output_path):
        try:
            temp_dir = os.path.dirname(output_path)
            temp_name = f"temp_{int(time.time() * 1000)}_{hashlib.md5(output_path.encode()).hexdigest()[:8]}.jpg"
            temp_path = os.path.join(temp_dir, temp_name)

            success = cv2.imwrite(temp_path, image, [cv2.IMWRITE_JPEG_QUALITY, 95])

            if success:
                try:
                    os.rename(temp_path, output_path)
                    return True
                except:
                    print(f"Rename failed, image saved to: {temp_path}")
                    return True
            return False
        except Exception as e:
            print(f"OpenCV save failed {output_path}: {str(e)}")
            return False

    def save_image(self, image, output_path):
        if self.save_image_pil(image, output_path):
            return True

        if self.save_image_cv2(image, output_path):
            return True

        ascii_path = self.convert_to_ascii_path(output_path)
        print(f"Trying ASCII path: {ascii_path}")
        return self.save_image_cv2(image, ascii_path)

    def convert_to_ascii_path(self, path):
        dir_name = os.path.dirname(path)
        file_name = os.path.basename(path)

        safe_name = re.sub(r'[^a-zA-Z0-9._-]', '_', file_name)
        safe_name = re.sub(r'_+', '_', safe_name)

        return os.path.join(dir_name, safe_name)

    def process_image(self, img_path, output_dir, image_index, variants=20):
        try:
            image = self.read_image(img_path)
            if image is None:
                return 0, []

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            generated_count = 0
            saved_paths = []

            base_name = os.path.basename(img_path)
            safe_base_name = self.sanitize_filename(base_name)

            for i in range(variants):
                augmented = self.transform_pipeline(image=image_rgb)["image"]

                timestamp = int(time.time() * 1000) % 1000000
                random_hash = hashlib.md5(f"{safe_base_name}{timestamp}{i}".encode()).hexdigest()[:8]

                output_filename = f"{safe_base_name}_{image_index:04d}_{i:02d}_{random_hash}.jpg"
                output_path = os.path.join(output_dir, output_filename)

                augmented_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)

                success = self.save_image(augmented_bgr, output_path)

                if success:
                    generated_count += 1
                    saved_paths.append(output_path)
                else:
                    print(f"Warning: Failed to save image {output_path}")

            return generated_count, saved_paths

        except Exception as e:
            print(f"Augmentation failed {img_path}: {str(e)}")
            return 0, []

    def augment_images(self, target_per_class=500, variants_per_image=20):
        os.makedirs(self.output_base_dir, exist_ok=True)

        total_generated = 0
        total_original = 0
        all_saved_paths = []

        for input_dir in self.input_dirs:
            if not os.path.exists(input_dir):
                print(f"Warning: Directory not found {input_dir}")
                continue

            class_name = os.path.basename(input_dir)
            class_output_dir = os.path.join(self.output_base_dir, class_name)

            if os.path.exists(class_output_dir):
                for file in os.listdir(class_output_dir):
                    file_path = os.path.join(class_output_dir, file)
                    try:
                        if os.path.isfile(file_path):
                            os.unlink(file_path)
                    except Exception as e:
                        print(f"Failed to delete file {file_path}: {e}")
            else:
                os.makedirs(class_output_dir, exist_ok=True)

            class_images = []
            for root, _, files in os.walk(input_dir):
                for file in files:
                    if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        img_path = os.path.join(root, file)
                        class_images.append(img_path)

            if not class_images:
                print(f"Warning: No images found in directory {input_dir}")
                continue

            print(f"\nProcessing class: {class_name}")
            print(f"Original images count: {len(class_images)}")

            if len(class_images) * variants_per_image < target_per_class:
                adjusted_variants = (target_per_class + len(class_images) - 1) // len(class_images)
                print(f"Adjusted variants per image: {variants_per_image} -> {adjusted_variants}")
                current_variants = adjusted_variants
            else:
                current_variants = variants_per_image

            class_generated = 0
            class_saved_paths = []
            progress_bar = tqdm(total=len(class_images), desc=f"Augmenting {class_name}")

            for idx, img_path in enumerate(class_images):
                generated, saved_paths = self.process_image(
                    img_path,
                    class_output_dir,
                    idx,
                    current_variants
                )
                class_generated += generated
                class_saved_paths.extend(saved_paths)
                progress_bar.update(1)

            progress_bar.close()

            actual_count = len([f for f in os.listdir(class_output_dir)
                              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])

            total_original += len(class_images)
            total_generated += class_generated
            all_saved_paths.extend(class_saved_paths)

            print(f"Class {class_name} completed:")
            print(f"  Original images: {len(class_images)}")
            print(f"  Claimed generated: {class_generated}")
            print(f"  Actual saved: {actual_count}")
            print(f"  Output directory: {class_output_dir}")

        print(f"\n{'='*60}")
        print("Final Verification Results:")
        print(f"{'='*60}")

        for class_name in sorted(os.listdir(self.output_base_dir)):
            class_dir = os.path.join(self.output_base_dir, class_name)
            if os.path.isdir(class_dir):
                files = [f for f in os.listdir(class_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                file_count = len(files)
                print(f"{class_name}/ - {file_count} images")

                if files:
                    print(f"  Sample files: {files[:5]}")

        print(f"{'='*60}")
        print(f"Total original images: {total_original}")
        print(f"Total generated images: {total_generated}")
        print(f"Total actual saved: {len(all_saved_paths)}")
        print(f"Output base directory: {self.output_base_dir}")

        log_file = os.path.join(self.output_base_dir, "augmentation_log.txt")
        with open(log_file, 'w', encoding='utf-8') as f:
            f.write(f"Augmentation Log - {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total original images: {total_original}\n")
            f.write(f"Total generated images: {total_generated}\n")
            f.write(f"Total actual saved: {len(all_saved_paths)}\n")
            f.write("\nSaved files:\n")
            for path in all_saved_paths[:100]:
                f.write(f"{path}\n")
            if len(all_saved_paths) > 100:
                f.write(f"... and {len(all_saved_paths) - 100} more files\n")
        print(f"Detailed log saved to: {log_file}")


if __name__ == "__main__":
    INPUT_DIRS = [
        r"E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\immature",
        r"E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\mature",
    ]

    OUTPUT_BASE_DIR = r"E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\augmented_dataset"

    augmenter = IndustrialCompostAugmenter(INPUT_DIRS, OUTPUT_BASE_DIR)

    augmenter.augment_images(
        target_per_class=500,
        variants_per_image=20
    )


Processing class: immature
Original images count: 89


Augmenting immature: 100%|█████████████████████████████████████████████████████████████| 89/89 [06:41<00:00,  4.51s/it]


Class immature completed:
  Original images: 89
  Claimed generated: 1780
  Actual saved: 1780
  Output directory: E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\augmented_dataset\immature

Processing class: mature
Original images count: 53


Augmenting mature: 100%|███████████████████████████████████████████████████████████████| 53/53 [03:57<00:00,  4.48s/it]

Class mature completed:
  Original images: 53
  Claimed generated: 1060
  Actual saved: 1060
  Output directory: E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\augmented_dataset\mature

Final Verification Results:
immature/ - 1780 images
  Sample files: ['25.4.17-CK新_0000_00_8b51848c.jpg', '25.4.17-CK新_0000_01_b63ef181.jpg', '25.4.17-CK新_0000_02_eba2a283.jpg', '25.4.17-CK新_0000_03_7f361ac6.jpg', '25.4.17-CK新_0000_04_7be37799.jpg']
mature/ - 1060 images
  Sample files: ['2025.5.11-CK新_0000_00_836560c7.jpg', '2025.5.11-CK新_0000_01_cd9fb693.jpg', '2025.5.11-CK新_0000_02_35bc5db0.jpg', '2025.5.11-CK新_0000_03_63af2f5c.jpg', '2025.5.11-CK新_0000_04_e84cdee4.jpg']
Total original images: 142
Total generated images: 2840
Total actual saved: 2840
Output base directory: E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\augmented_dataset
Detailed log saved to: E:\TSG\jupyterlab\machine learning image\Transfer Learning\Classification\augmented_datase